# 02 — Profile builder

**What this tests:** documents → structured fields → graded against ground truth →
merged into one `Profile`.

- Extraction is **rules first, LLM second**: regex finds e-mails/phones/links for free,
  then the LLM (Gemini Flash or the offline fake) fills the schema.
- Grading is **case-insensitive and whitespace-normalized** (`"  MSc   Data Science "`
  == `"msc data science"`), so formatting never fails a comparison.
- With `LLM_PROVIDER=fake` everything runs offline on canned data: expect **low**
  accuracies (canned ≠ ground truth) but valid schemas. With real Gemini the scores
  should be high — that gap is exactly how you spot a bad extractor or prompt.

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import sys
from pathlib import Path

# Make oppradar_ai importable whether the kernel starts in ai/ or ai/notebooks/.
AI_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

from oppradar_ai.config import get_settings, setup_logging
from oppradar_ai.documents import (
    build_profile,
    extract_certificate,
    extract_cv,
    extract_portfolio,
    grade_extraction,
    read_document,
)

setup_logging('INFO')
settings = get_settings().ensure_dirs()
SAMPLES = settings.samples_dir
TRUTH = SAMPLES / 'ground_truth'
print('provider:', settings.llm_provider)

## Step 1 — Extract a CV

Note the e-mail: the fake LLM returns an empty e-mail on purpose, so you see the
**regex backfill** fill it from the raw text — rules before LLM, always.

In [ ]:
cv_doc = read_document(SAMPLES / 'cv_amine.pdf')
cv = extract_cv(cv_doc.text, cv_doc.file_name)

print('name  :', cv.full_name)
print('email :', cv.email, ' <- regex backfill')
print('skills:', cv.skills[:10])
print('links :', cv.links)
print('review:', cv.needs_review or 'nothing flagged')

## Step 2 — Grade it against the ground truth

`ground_truth/cv_amine.json` holds the fields a perfect extraction would return.
`grade_extraction` compares normalized sets for lists and normalized strings for
scalars, and reports exactly which expected values are missing.

In [ ]:
truth = json.loads((TRUTH / 'cv_amine.json').read_text(encoding='utf-8'))
grade = grade_extraction(cv_doc.file_name, cv, truth)

print(f'accuracy: {grade.accuracy:.0%}  ({grade.fields_matched}/{grade.fields_checked} fields)')
print('per field:', grade.per_field)
if grade.missing:
    print('missing:')
    for item in grade.missing:
        print('  -', item)

## Step 3 — Certificate and portfolio

Same pattern for the other document types. The certificate's credential URL and the
portfolio's contact e-mail come from the regex rules; the structure comes from the LLM.

In [ ]:
cert_doc = read_document(SAMPLES / 'certificate_alx_amine.pdf')
cert = extract_certificate(cert_doc.text, cert_doc.file_name)
cert_truth = json.loads((TRUTH / 'certificate_alx_amine.json').read_text(encoding='utf-8'))
cert_grade = grade_extraction(cert_doc.file_name, cert, cert_truth)

port_doc = read_document(SAMPLES / 'portfolio_sara.docx')
port = extract_portfolio(port_doc.text, port_doc.file_name)
port_truth = json.loads((TRUTH / 'portfolio_sara.json').read_text(encoding='utf-8'))
port_grade = grade_extraction(port_doc.file_name, port, port_truth)

print(f'certificate: {cert.title} — {cert.issuer} (url={cert.credential_url})')
print(f'  accuracy {cert_grade.accuracy:.0%}')
print(f'portfolio  : {len(port.projects)} projects, email={port.email}')
print(f'  accuracy {port_grade.accuracy:.0%}')

## Step 4 — Build Amine's profile (CV + certificate)

`build_profile` unions the lists (case-insensitive dedupe), keeps the first non-empty
scalar, records **conflicts** when documents disagree, scores **completeness** over
10 key sections, and turns everything uncertain into low-confidence fields to verify.

In [ ]:
result = build_profile([cv, cert], user_id='u-amine')
profile = result.profile

print('merged from :', result.merged_from)
print('completeness:', f'{result.completeness:.0%}')
print('skills      :', profile.skills)
print('certificates:', [c.title for c in profile.certificates])
print('conflicts   :', result.conflicts or 'none')
print('to verify   :', [(f.field, f.confidence) for f in profile.to_verify()] or 'nothing')
for suggestion in result.suggestions:
    print('suggestion  :', suggestion)

## Step 5 — Build Sara's profile (scanned CV + portfolio)

Sara's CV only exists as a scan, so this profile starts from OCR text — the reader's
warning and the OCR method stay attached to the source file for the review queue.

In [ ]:
scanned = read_document(SAMPLES / 'cv_sara_scanned.pdf')
sara_cv = extract_cv(scanned.text, scanned.file_name)
sara = build_profile([sara_cv, port], user_id='u-sara')

print('name        :', sara.profile.full_name)
print('headline    :', sara.profile.headline)
print('completeness:', f'{sara.completeness:.0%}')
print('skills      :', sara.profile.skills)
print('projects    :', [p.name for p in sara.profile.projects])
print('conflicts   :', sara.conflicts or 'none')

## Summary

- `read_document` → `extract_*` → `grade_extraction` → `build_profile` is the whole
  document pipeline, and every step returns a Pydantic-validated object.
- Grading is honest about *which* fields failed and never trips on case or spacing.
- Conflicts and OCR provenance end up in `field_confidences` / `to_verify()` so the UI
  can ask the user to confirm instead of guessing.

**Next:** notebook `03_ingestion.ipynb` finds opportunities on the live web.